# Microbiome Single-Ignition EI

This notebook tests whether single-node effective information (EI) from ignition strength to whole-system final mean activation is more informative about ignition success than the paper's tree-reach recovery ranking.


## Question and data

We reuse the microbiome reproduction cache from `results/network_revival_microbiome/`. The source variable is `Delta_i ~ Uniform(0, 10)` for one fixed ignition node. The target is the final whole-system mean activation after ecological forcing with the same no-release protocol used in the paper reproduction.

The comparison baseline is the paper-style recovery table: `TreeSize`, final `State`, paper rank, and binary `success = State > 5`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'network_revival':
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == 'exp':
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from exp.network_revival.microbiome import MicrobiomeParameters
from exp.network_revival.microbiome_ei import (
    DEFAULT_OUTPUT_DIR,
    MicrobiomeEIConfig,
    load_microbiome_ei_inputs,
    plot_microbiome_ei_comparison,
    run_microbiome_single_node_ei,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## Configuration

The default cell runs the full cache-first experiment (`568 x 32` ODE solves). For a quick smoke run, set `SMOKE = True`; this evaluates only three nodes and writes to a separate smoke directory.


In [ ]:
SMOKE = False
FORCE_RECOMPUTE = False

base_params = MicrobiomeParameters(dt=0.05, t_max=20.0)
if SMOKE:
    ei_config = MicrobiomeEIConfig(
        n_delta=5,
        delta_max=10.0,
        seed=42,
        target_noise_fraction=0.01,
        node_indices=(0, 1, 2),
        output_dir=REPO_ROOT / 'results/network_revival_microbiome_ei_smoke',
        params=MicrobiomeParameters(dt=0.2, t_max=0.6),
    )
else:
    ei_config = MicrobiomeEIConfig(
        n_delta=32,
        delta_max=10.0,
        seed=42,
        target_noise_fraction=0.01,
        output_dir=REPO_ROOT / 'results/network_revival_microbiome_ei',
        params=base_params,
    )

ei_config


## Run or load EI cache

The runner writes sample arrays, EI summaries, comparison tables, and a manifest under `results/network_revival_microbiome_ei/`. With `force_recompute=False`, existing cache files are loaded without recomputing the ODE sweep.


In [ ]:
inputs = load_microbiome_ei_inputs(REPO_ROOT / 'results/network_revival_microbiome')
print('Active adjacency:', inputs['adjacency'].shape)
print('Recovery rows:', len(inputs['recovery_ranked']))

ei_result = run_microbiome_single_node_ei(
    ei_config,
    adjacency=inputs['adjacency'],
    active_indices=inputs['active_indices'],
    recovery_ranked=inputs['recovery_ranked'],
    force_recompute=FORCE_RECOMPUTE,
)

print('Samples:', ei_result['mean_response_samples'].shape)
print('Summary CSV:', ei_result['cache_paths']['summary_csv'])
print('Comparison CSV:', ei_result['cache_paths']['comparison_csv'])
print('Manifest:', ei_result['cache_paths']['manifest_json'])


## Ranking comparison

We compare EI against the paper recovery ranking using rank correlations, success-prediction metrics, and top-k enrichment.


In [ ]:
comparison = ei_result['comparison'].copy()
metrics = ei_result['metrics']

metric_rows = pd.DataFrame(
    [{'metric': key, 'value': value} for key, value in sorted(metrics.items())]
)
display(metric_rows)

cols = [
    'active_node', 'species_index', 'species_name', 'ei_rank', 'ei_mean_response',
    'rank', 'tree_size', 'state', 'success', 'ei_residual_vs_tree_size'
]
available_cols = [col for col in cols if col in comparison.columns]
display(comparison.sort_values('ei_rank')[available_cols].head(20))


## Figures

The helper writes all figures into the EI results directory. Figure labels are in English and legends are placed outside axes where needed.


In [ ]:
figure_paths = plot_microbiome_ei_comparison(
    comparison,
    metrics,
    ei_config.output_dir,
    k_values=(10, 20, 50),
)

for name, figure_path in figure_paths.items():
    print(name, figure_path)
    display(Image(filename=str(figure_path)))


## Interpretation checklist

Use these outputs to decide whether EI reveals ignition success better than the paper metric:

- `ei_success_auroc` and `ei_success_average_precision` should exceed the corresponding `tree_size_*` values if EI is more predictive.
- `ei_precision_at_K` vs `tree_size_precision_at_K` tests whether the top-ranked nodes are enriched for successful igniters.
- The EI residual figure highlights nodes whose information score is high after removing the linear trend with `TreeSize`; successful high-residual nodes are candidates where EI adds explanatory value beyond the paper's reach definition.
